# Custom Strategies

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/douglasrizzo/catsim/blob/main/notebooks/04_custom_strategies.ipynb)

**For:** method authors who implement a selection, stopping or exposure-control method that catsim
lacks. **Level:** advanced. You know item response theory and Python classes, and have run a
simulation, as the [getting-started notebook](01_getting_started.ipynb) does.

You will:

- write a selector, a stopper and an exposure control by subclassing catsim's base classes;
- read what a strategy gets from its step context and from an estimate;
- call a strategy directly on a session, which is how to test one;
- run each strategy in a simulation, like the built-in ones;
- see why a strategy that runs on several processes must live in a module.

The [Extending catsim](https://douglasrizzo.com.br/catsim/extending.html) guide writes one strategy
of each kind and says what each must do. To run the notebook outside this repository, install
catsim first: `pip install catsim`.

In [ ]:
import dataclasses
from dataclasses import dataclass
from typing import override

import numpy
from irtlib import ItemBank

from catsim import CatEngine, StepContext, simulate
from catsim.estimation import BaseEstimator, EAPEstimator
from catsim.exposure import BaseExposureControl
from catsim.initialization import FixedPointInitializer
from catsim.selection import BaseSelector, MaxInfoSelector, best_pool, pick_best, remaining
from catsim.stopping import BaseStopper, MaxLengthStopper, StopRule

## A bank and a population

A generated bank of `N_ITEMS` items under the three-parameter logistic model, and `N_EXAMINEES`
simulated examinees whose abilities are evenly spread from −2.5 to 2.5, so that every part of the
scale has examinees. Every engine here starts at an ability estimate of 0 and estimates ability by
expected a posteriori (EAP); `engine_with` builds one from a selector, a stopper and, optionally, an
exposure control. `BANK_SEED` draws the bank, and `RUN_SEED` seeds every run: the examinees' answers
and every random draw a strategy makes.

In [ ]:
BANK_SEED = 11
RUN_SEED = 5
N_ITEMS = 300
N_EXAMINEES = 200

item_bank = ItemBank.generate(N_ITEMS, model="3PL", rng=BANK_SEED)
true_thetas = numpy.linspace(-2.5, 2.5, N_EXAMINEES)
estimator = EAPEstimator()


def engine_with(
  selector: BaseSelector, stopper: BaseStopper, exposure_control: BaseExposureControl | None = None
) -> CatEngine:
  """Build an engine that starts at 0 and estimates by EAP, with the given selector, stopper and control."""
  return CatEngine(
    item_bank, FixedPointInitializer(0.0), selector, estimator, stopper, exposure_control=exposure_control
  )

## A custom selector

Under maximum-information selection, every examinee gets the same first item, since all start at the
same estimate, and that item's exposure rate is 1. The first items are chosen with a rough estimate
anyway, so the item most informative at the estimate may not be the most informative at the
examinee's ability. `EarlyRandomesqueSelector` draws at random among the `pool_size` most
informative items while the standard error is above `until_se`, and gives the most informative item
after that.

A selector subclasses `BaseSelector` and overrides `select`, which receives a `StepContext` and
returns an item index. This one reads the item bank, the estimate `theta` and its standard error
`se` from the context; `remaining`, `best_pool` and `pick_best` are the helpers the built-in
selectors use
([Extending catsim](https://douglasrizzo.com.br/catsim/extending.html#a-selector)). Its
random draws come from `context.rng`, the generator of the call. `@override`, from `typing`, lets a
type checker report a method that overrides nothing. The class is a frozen dataclass: its fields are its configuration, and it cannot
store anything about a session. One selector serves every session of an engine, so a value it stored
for one session would be read by the next.

In [ ]:
@dataclass(frozen=True)
class EarlyRandomesqueSelector(BaseSelector):
  """Draw among the most informative items while the estimate is rough, then give the most informative."""

  pool_size: int
  until_se: float

  @override
  def select(self, context: StepContext) -> int:
    candidates = remaining(context)
    information = context.item_bank.information(context.theta)[candidates]
    if context.se <= self.until_se:
      return pick_best(candidates, information, context.rng)
    return int(context.rng.choice(best_pool(candidates, information, self.pool_size)))


POOL_SIZE = 10
UNTIL_SE = 0.7
TEST_LENGTH = 15

selector = EarlyRandomesqueSelector(POOL_SIZE, UNTIL_SE)
engine = engine_with(selector, MaxLengthStopper(TEST_LENGTH))

## Call it on a session

A strategy can be called directly, which is how to test one. Here an examinee answers two items,
both correctly, and `engine.context(session)` returns the step context the selector would get for
the session's next item. The selector is then asked 50 times what to give next.

In [ ]:
rng = numpy.random.default_rng(RUN_SEED)
session = engine.start(rng=rng)
for response in (True, True):
  selection = engine.select(session, rng=rng)
  session = engine.answer(selection.session, selection.item, response, rng=rng)

context = engine.context(session, rng=rng)
print(context.administered_items, context.responses, context.excluded_items, context.exposure_rates)
print(session.estimates[-1])
print(sorted({selector.select(context) for _ in range(50)}), len(session.administered_items))

The context holds the two items given and their responses, no excluded items, and no exposure rates,
which only a caller that counts exposure passes. Its `theta` and `se` are those of the session's
newest `Estimate`: the ability estimate, 0.52, its standard error, 0.88, and `evaluations`, how many
times the estimator evaluated its objective. The standard error is above `UNTIL_SE`, so the selector
draws: 50 calls give only 10 different items, the pool of the 10 most informative. Calling a
strategy changes nothing in the session, which still has two items.

## Run it in a simulation

The selector goes wherever a built-in one does. Here it runs tests of `TEST_LENGTH` items, against
`MaxInfoSelector`, on the same seed, so that every examinee gives the same answer to an item in both
runs.

In [ ]:
max_info_engine = engine_with(MaxInfoSelector(), MaxLengthStopper(TEST_LENGTH))
max_info = simulate(max_info_engine, true_thetas=true_thetas, rng=RUN_SEED)
early = simulate(engine, true_thetas=true_thetas, rng=RUN_SEED)
for name, result in (("max info", max_info), ("early randomesque", early)):
  first_items = {session.administered_items[0] for session in result.sessions}
  rmse = result.rmse()
  print(
    f"{name}: first items {len(first_items)}, highest exposure rate"
    f" {result.exposure_rates().value.max():.2f}, RMSE {rmse.value:.3f} ± {rmse.mc_se:.3f}"
  )
drawn = [sum(estimate.se > UNTIL_SE for estimate in session.estimates[:-1]) for session in early.sessions]
print(f"items drawn at random per test: {numpy.mean(drawn):.1f}")

again = simulate(engine, true_thetas=true_thetas, rng=RUN_SEED)
print(again.sessions == early.sessions)

Maximum information gives all 200 examinees the same first item, at an exposure rate of 1.00. The
custom selector spreads the first items over 10, its pool, and the highest exposure rate falls to
0.69. A session's items are selected with the estimates before its last, so counting those above
`UNTIL_SE` gives the items drawn at random: 4.8 per test on average. The two RMSEs, 0.473 ± 0.026
and 0.470 ± 0.024, differ by far less than their Monte Carlo errors, so this run does not rank the
selectors; the [comparing-strategies notebook](02_comparing_strategies.ipynb) compares designs by
their paired differences.

The second run on the same seed gives every examinee the same test: every random draw came from the
generator of the call. A strategy that draws from anywhere else, such as `numpy.random`'s global
functions or a generator of its own, escapes the run's seed
([Reproducibility](https://douglasrizzo.com.br/catsim/reproducibility.html)).

## A custom stopper

`EstimateChangeStopper` ends the test once the newest response moves the ability estimate by less
than `max_change`: more items would change the result little. A stopper subclasses `BaseStopper` and
overrides `stop`, which receives the context with the estimate updated by the newest response as
`theta`.

The estimate before the newest response is not in the context, and the stopper must not keep it: one
stopper serves every session of an engine, so a stored estimate would belong to whichever session
called last. The stopper recomputes it: it gives its estimator the context without the newest item
and response, built with `dataclasses.replace`, and reads `theta` from the `Estimate` it returns.
That context still holds the newest estimate as `theta`. An estimator that does not start from it,
such as the EAP estimator used here, recomputes exactly the estimate the session recorded; one that
starts its search from it can give a different value. With one item, the estimate before it came
from the initializer, which the stopper cannot recompute, so it waits for a second item.

A `StopRule` combines the custom stopper with a cap of `MAX_ITEMS`, so that every test ends even if
the estimate keeps moving.

In [ ]:
@dataclass(frozen=True)
class EstimateChangeStopper(BaseStopper):
  """Stop once the newest response moves the ability estimate by less than `max_change`."""

  estimator: BaseEstimator
  max_change: float

  @override
  def stop(self, context: StepContext) -> bool:
    if len(context.administered_items) <= 1:
      return False
    before = dataclasses.replace(
      context, administered_items=context.administered_items[:-1], responses=context.responses[:-1]
    )
    return abs(context.theta - self.estimator.estimate(before).theta) < self.max_change


MAX_CHANGE = 0.05
MAX_ITEMS = 30

change = EstimateChangeStopper(estimator, MAX_CHANGE)
rule = StopRule(change, MaxLengthStopper(MAX_ITEMS))
change_engine = engine_with(MaxInfoSelector(), rule)
result = simulate(change_engine, true_thetas=true_thetas, rng=RUN_SEED)

examinees = result.examinees()
lengths = examinees["test_length"]
print(f"test lengths from {lengths.min()} to {lengths.max()}, mean {lengths.mean():.1f}")
held = numpy.array([change.stop(change_engine.context(session)) for session in result.sessions])
print(numpy.count_nonzero(held), numpy.unique(lengths[~held]))
print(f"final standard errors from {examinees['se'].min():.2f} to {examinees['se'].max():.2f}")

Test lengths range from 13 to 30 items, 20.9 on average. Called on each finished session, the
stopper holds for 197 of the 200; the other 3 have 30 items: they reached the cap with the estimate
still moving. A small change of the estimate is not a small standard error: the final standard
errors range from 0.33 to 0.48. To end tests at a target precision, combine the custom stopper with
a `StandardErrorStopper` in the rule.

## A custom exposure control

An exposure control decides, after the selector, whether to administer the item it proposed. The
engine asks it about every proposal; an item it declines is excluded from the session for the rest
of its test, and the selector proposes again. `HardCap` declines every item whose exposure rate has
reached `r_max`, so, unlike a selector's own cap, which gives a capped item when no other is left,
it never gives one. It accepts every item when the context has no exposure rates, as for the first
examinee of a run. In a simulation, each proposal is an administration or a decline, and the
result's `proposal_rates` count both.

In [ ]:
@dataclass(frozen=True)
class HardCap(BaseExposureControl):
  """Decline every item whose exposure rate has reached `r_max`."""

  r_max: float

  @override
  def accept(self, context: StepContext, item: int) -> bool:
    return context.exposure_rates is None or bool(context.exposure_rates[item] < self.r_max)


R_MAX = 0.2

capped_engine = engine_with(MaxInfoSelector(), MaxLengthStopper(TEST_LENGTH), HardCap(R_MAX))
capped = simulate(capped_engine, true_thetas=true_thetas, rng=RUN_SEED)
exposure, proposals = capped.exposure_rates().value, capped.proposal_rates().value
print(f"highest exposure rate {exposure.max():.2f}, highest proposal rate {proposals.max():.2f}")
declined = [len(session.excluded_items) for session in capped.sessions]
print(f"items declined per test: {numpy.mean(declined):.1f} on average, {max(declined)} at most")

No item's exposure rate ends above 0.20, while the highest proposal rate is 1.00: the selector
proposes its favourite item to every examinee, and the control declines it once its rate has reached
the cap. A declined item joins the session's excluded items, so each test declined 21.2 items on
average, and one declined 75 before it had all its items.

## Strategies on workers

`simulate(..., parts=..., workers=...)` runs the parts of a run on several processes. Each process
starts afresh and receives the engine pickled, which names every class by its module, and a class
defined in a notebook belongs to the notebook's main module, which the processes cannot import.
`simulate` checks this before any part runs:

In [ ]:
try:
  simulate(engine, true_thetas=true_thetas, rng=RUN_SEED, parts=2, workers=2)
except ValueError as error:
  print(error)

The fix is to move the strategy classes into a module, such as a file `my_strategies.py` beside the
notebook, and import them from it: `from my_strategies import EarlyRandomesqueSelector`. The worker
processes then import the module as the notebook does. The same holds for a lambda, such as a custom
prior written inline, and for a class defined inside a function. With `workers=1`, the default, a
run stays in the notebook's process, and any strategy runs. A script that runs on workers calls
`simulate` under `if __name__ == "__main__":`, as every process pool of Python requires.

## Where to go next

- The [Extending catsim](https://douglasrizzo.com.br/catsim/extending.html) guide writes an
  initializer, an estimator, a selector and a stopper, and says what each kind must do and the rules
  every strategy keeps.
- The [comparing-strategies notebook](02_comparing_strategies.ipynb) compares designs by their
  paired differences with their Monte Carlo errors, which is how to compare a custom strategy with
  the built-in ones.